# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Predicts the CTR-opportunity rank for the **same 15,310-item slice** as ML-05 and ML-07, using the **same honest features** (only what exists *before* the label window closes) and the **same `at_risk` label**.
The Week-4 rule baseline is re-computed in this notebook so baseline and model share data, metric, and split.

How this runs: the first cell builds the feature vector from the warehouse (one 2-month scan) and caches it to `work/outputs/w05_feature_vector.parquet`; later runs load the cache instead of re-scanning.

In [1]:
import importlib
if any(importlib.util.find_spec(m) is None for m in ("duckdb", "huggingface_hub", "sklearn")):
    get_ipython().run_line_magic("pip", "-q install duckdb huggingface_hub scikit-learn")

In [2]:
import os
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('hf_key')
except Exception:
    HF_TOKEN = os.getenv('hf_key')
if not HF_TOKEN:
    raise RuntimeError('hf_key not found: set the Colab secret or the hf_key env var')

In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':   f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':   f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':    f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Task shape:** *"which page first?"* — a ranking problem. We rank pages by **predicted probability that the page's click-through rate is below its slice median over the last 30 days** (`at_risk`), measured at precision@K, exactly the metric ML-07 used on the rule baseline.

**Method ladder (readable → stronger):**
1. **Logistic Regression** — its coefficients are a readable decision surface: do demand, exposure, and intent point up or down? We'll start here because the side-effect is interpretability, not because it is the best possible model.
2. **Random Forest (classifier)** — same features, nonlinear interactions allowed. Kept because the features are few and lazy (two categorical, two counts), so a forest is cheap and honest; we'll add complexity only if the comparison earns it.
3. **Random Forest (regressor on `ctr_label`)** — keeps the RMSE framing from ML-03 honest: a model that merely predicts the mean gets **0.3519**; the regressor's held-out RMSE is reported next to that number.

Only **pre-window features** enter: `impressions_prev30d` (days −60…−31), `search_volume`, `main_intent`, `content_type` — the same set ML-05 proved leak-free (Demo C/D). The regressions above are trained on the same folds the baseline is scored on.

In [4]:
import numpy as np
import pandas as pd
from pathlib import Path

repo = Path.cwd()
while not (repo / "work").exists() and repo != repo.parent:
    repo = repo.parent

bounds_sql = "DATE '2025-08-31' AS end_d"

def build_feature_vector(con, tables):
    return con.sql(f"""
    WITH bounds AS (
        SELECT {bounds_sql}
    ),
    windowed AS (
        SELECT
            q.client_hash_id,
            q.content_hash_id,
            ANY_VALUE(d.main_intent) AS main_intent,
            ANY_VALUE(d.content_type) AS content_type,
            ANY_VALUE(d.search_volume) AS search_volume,
            SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 60 DAY AND q.report_date <= b.end_d - INTERVAL 30 DAY
                    THEN q.gsc_impressions ELSE 0 END) AS impressions_prev30d,
            SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY THEN q.gsc_impressions ELSE 0 END) AS impressions_last30d,
            SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY THEN q.gsc_clicks ELSE 0 END) AS clicks_30d,
            AVG(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY THEN q.gsc_avg_position END) AS avg_position_30d,
            100.0 *
            COALESCE(CAST(SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY
                            THEN q.gsc_clicks ELSE 0 END) AS DOUBLE) /
                NULLIF(SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY
                            THEN q.gsc_impressions ELSE 0 END), 0), 0) AS ctr_label
        FROM {tables['fact_daily']} q
        CROSS JOIN bounds b
        JOIN {tables['dim_content']} d
          ON q.content_hash_id = d.content_hash_id
        WHERE q.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY q.client_hash_id, q.content_hash_id
        HAVING SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 60 DAY
                     AND q.report_date <= b.end_d - INTERVAL 30 DAY
                    THEN q.gsc_impressions ELSE 0 END) >= 70
    )
    SELECT * FROM windowed
    """).df()

cache_f = repo / "work" / "outputs" / "w05_feature_vector.parquet"
if cache_f.exists():
    df = pd.read_parquet(cache_f)
    print("loaded cached feature vector:", cache_f)
else:
    df = build_feature_vector(con, TABLES)
    cache_f.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(cache_f, index=False)
    print("wrote cache:", cache_f)

# at-risk label, exactly as ML-07 defined it (ctr below the slice median over last 30 days)
midline = df["ctr_label"].median()
df["at_risk"] = (df["ctr_label"] < midline).astype(int)

print(f"{len(df):,} content items with enough history")
print("base rate (share at-risk in slice):", f"{df['at_risk'].mean():.3f}")
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

wrote cache: /work/outputs/w05_feature_vector.parquet
15,310 content items with enough history
base rate (share at-risk in slice): 0.500


,client_hash_id,content_hash_id,main_intent,content_type,search_volume,impressions_prev30d,impressions_last30d,clicks_30d,avg_position_30d,ctr_label,at_risk
0,client_73cda7b4e4f265ea,content_7476254360f720e6,commercial,keyword article,1600,1763.0,5576.0,11.0,29.719714,0.197274,0
1,client_73cda7b4e4f265ea,content_74a2f5963ed23ca4,commercial,keyword article,30,2711.0,40140.0,106.0,3.773460,0.264076,0
2,client_73cda7b4e4f265ea,content_74e8bcbda51fa8d5,transactional,keyword article,10,302.0,52269.0,649.0,3.287721,1.241654,0
3,client_73cda7b4e4f265ea,content_750a5b5bf1a8f283,informational,keyword article,20,165.0,519.0,0.0,37.298052,0.000000,1
4,client_73cda7b4e4f265ea,content_753ccbd05318c4d0,informational,keyword article,10,179.0,1879.0,1.0,12.375078,0.053220,1


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by client, 5-fold.** Every content item belongs to exactly one client; `GroupKFold(n_splits=5)` on `client_hash_id` puts whole clients in the test fold, so each score is a prediction for a client the model **never saw during training**.

Why grouped rather than random: ML-05 Demo D measured the cost of a random split — letting the forest memorize per-client habits inflated skill by **+0.056 RMSE** versus grouping. The decision we serve is "which page on *this* client should be reviewed first," so an honest split must not assume the new client's quirks were in the training data.

The rule baseline is scored on the **same folds**, by the same precision@K: top-K pages within each held-out fold, judged by whether `at_risk = 1`. Same slice (15,310 rows), same label, same metric — only the scorer changes.

- Label window: 2025-08-01 → 2025-08-31 (`ctr_label`). Features use only days before it.
- `random_state = 42` everywhere; seeds stated, so re-running reproduces the table.

In [5]:
from sklearn.model_selection import GroupKFold

Y = df[["at_risk", "ctr_label", "client_hash_id"]].copy()
CV, SEED = 5, 42

# honest feature matrix — missing flags first, THEN fill (no blind fillna(0))
X = df[["search_volume", "impressions_prev30d", "main_intent", "content_type"]].copy()
X["has_search_volume"] = X["search_volume"].notna().astype(int)
X["has_main_intent"] = X["main_intent"].notna().astype(int)
X["search_volume"] = X["search_volume"].fillna(0)
X["main_intent"] = X["main_intent"].fillna("unknown")
X["content_type"] = X["content_type"].fillna("unknown")
X = pd.get_dummies(X, columns=["main_intent", "content_type"], dtype=int)
feature_cols = list(X.columns)

y_risk = df["at_risk"].to_numpy()
y_ctr = df["ctr_label"].to_numpy()
groups = df["client_hash_id"].to_numpy()

## spliting using GroupKfold
gkf = GroupKFold(n_splits=CV)
folds = list(gkf.split(X, y_risk, groups))
for tr, te in folds:
    assert not set(groups[tr]) & set(groups[te]), "client overlap between train and test"
print("client overlap across folds: none (grouped split holds)")
print("train/test rows per fold:", [(len(t), len(v)) for t, v in folds])
print("clients in slice:", df["client_hash_id"].nunique())
print("feature columns:", feature_cols)
print("held-out clients per fold:")
for i, (tr, te) in enumerate(folds):
    print(f"  fold {i}: {len(set(groups[te]))} test clients")

client overlap across folds: none (grouped split holds)
train/test rows per fold: [(7575, 7735), (12356, 2954), (12613, 2697), (14293, 1017), (14403, 907)]
clients in slice: 14
feature columns: ['search_volume', 'impressions_prev30d', 'has_search_volume', 'has_main_intent', 'main_intent_commercial', 'main_intent_informational', 'main_intent_navigational', 'main_intent_transactional', 'main_intent_unknown', 'content_type_feedly article', 'content_type_keyword article']
held-out clients per fold:
  fold 0: 1 test clients
  fold 1: 1 test clients
  fold 2: 1 test clients
  fold 3: 1 test clients
  fold 4: 10 test clients


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The Week-4 rule is re-built in this notebook (no fitted weights) so both scorers are measured in the **same run**:

```text
baseline_action_score =
  0.50 * visibility_score  (percentile of log impressions_prev30d)
+ 0.45 * demand_score      (percentile of log search_volume, 0 if missing)
+ 0.05 * transactional_flag
```

Honest expectation going in: ML-07 measured only weak point-biserial correlations between these features and `at_risk` (|r| ≈ 0.04–0.10). So a *modest* gain — or a loss at some K — is the likely, honest outcome. We report both kinds of cells in the table rather than picking the favorable K.

**Verdict columns read left to right:** precision@K (higher = more of the top of the queue truly low-CTR) and RF-regressor RMSE vs the predict-the-mean RMSE 0.3519.

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import mean_squared_error

def precision_at_k(score, y, k):
    order = np.argsort(-np.asarray(score), kind="stable")
    return float(np.asarray(y)[order[:k]].mean())

# baseline rule (same as ML-07, recomputed here)
df["visibility_score"] = df["impressions_prev30d"].rank(pct=True)  # log not needed at fold level
df["has_sv"] = df["search_volume"].notna()
df["demand_score"] = np.log1p(df["search_volume"].fillna(0)).rank(pct=True) * df["has_sv"]
df["baseline_action_score"] = (
    0.50 * df["visibility_score"]
    + 0.45 * df["demand_score"]
    + 0.05 * (df["main_intent"] == "transactional")
).clip(0, 1)

base_rmse = float(np.sqrt(np.mean((y_ctr - y_ctr.mean()) ** 2)))

KS = (10, 20, 50)
fold_scores = {"baseline": [], "lr": [], "rf": []}
fold_rmse_rf = []
for tr, te in folds:
    fold_scores["baseline"].append(df["baseline_action_score"].to_numpy()[te])

    lr = LogisticRegression(max_iter=2000, random_state=SEED)
    lr.fit(X.iloc[tr], y_risk[tr])
    fold_scores["lr"].append(lr.predict_proba(X.iloc[te])[:, 1])

    rf = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
    rf.fit(X.iloc[tr], y_risk[tr])
    fold_scores["rf"].append(rf.predict_proba(X.iloc[te])[:, 1])

    rfr = RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1)
    rfr.fit(X.iloc[tr], y_ctr[tr])
    fold_rmse_rf.append(float(np.sqrt(mean_squared_error(y_ctr[te], rfr.predict(X.iloc[te])))))

rows = [["base rate"] + [round(float(y_risk.mean()), 3)] * 3 + [f"{base_rmse:.4f}"]]
for name in ("baseline", "lr", "rf"):
    p = [round(float(np.mean([precision_at_k(fold_scores[name][i], y_risk[folds[i][1]], k)
                               for i in range(CV)])), 3) for k in KS]
    rows.append([name] + p + ["--"])
rows.append(["rf_reg(ctr)"] + ["--"] * 3 + [f"{float(np.mean(fold_rmse_rf)):.4f}"])

table = pd.DataFrame(rows, columns=["scorer", "p@10", "p@20", "p@50", "rmse_ctr"])
print(table.to_string(index=False))

     scorer  p@10  p@20   p@50 rmse_ctr
  base rate   0.5   0.5    0.5   0.3519
   baseline   0.6  0.62  0.584       --
         lr  0.86  0.83  0.844       --
         rf  0.62  0.61  0.608       --
rf_reg(ctr)    --    --     --   0.4174


### What the table says

- **Base rate** is the floor precision@K: `at_risk` is balanced by construction (`ctr < median`), so a coin flip scores 0.500. The shared RMSE floor (predict the mean) is 0.3519.
- **Baseline rule** is the ML-07 rule scored on the same folds — the opponent the model must beat.
- **LR** is the readable logistic model; **RF** is the same features with a forest.
- `rf_reg(ctr)` is the RMSE framing: does predicting CTR beat the mean?

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Metric first, errors second — but errors are the point. Three questions:

1. **What does the model lean on?** Permutation importance (drop in held-in AUC when a column is shuffled) on the full honest feature set. Then sanity-check the top feature: does it *plausibly* relate to low CTR, or is it suspiciously perfect (= leak)?
2. **Where is it most wrong?** False reviews (model says at-risk, page actually fine — wasted review time) and misses (page actually at-risk, model ranked it low).
3. **Why are those cases hard?** Say it in plain words, with the observed facts.

In [17]:
from sklearn.inspection import permutation_importance

lr = LogisticRegression(max_iter=2000, random_state=SEED)
lr.fit(X, y_risk)

imp = permutation_importance(lr, X, y_risk, n_repeats=5, random_state=SEED,
                             scoring="roc_auc", n_jobs=-1)
fi = pd.DataFrame({"feature": feature_cols,
                   "auc_drop": imp.importances_mean,
                   "std": imp.importances_std}).sort_values("auc_drop", ascending=False)
print("Permutation importance for Logistic Regressor model")
print("permutation importance on at_risk (AUC drop when shuffled; within-sample, interpretation only):")
print(fi.round(4).to_string(index=False))

Permutation importance for Logistic Regressor model
permutation importance on at_risk (AUC drop when shuffled; within-sample, interpretation only):
                     feature  auc_drop    std
               search_volume    0.0694 0.0042
         impressions_prev30d    0.0564 0.0036
   main_intent_transactional    0.0174 0.0029
           has_search_volume    0.0080 0.0020
 content_type_feedly article    0.0010 0.0004
         main_intent_unknown    0.0008 0.0004
content_type_keyword article    0.0005 0.0002
             has_main_intent    0.0004 0.0003
    main_intent_navigational    0.0000 0.0000
      main_intent_commercial   -0.0001 0.0001
   main_intent_informational   -0.0004 0.0006


In [18]:
from sklearn.inspection import permutation_importance

rf_full = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
rf_full.fit(X, y_risk)

imp = permutation_importance(rf_full, X, y_risk, n_repeats=5, random_state=SEED,
                             scoring="roc_auc", n_jobs=-1)
fi = pd.DataFrame({"feature": feature_cols,
                   "auc_drop": imp.importances_mean,
                   "std": imp.importances_std}).sort_values("auc_drop", ascending=False)
print("Permutation importance for Random Forest Classifier")
print("permutation importance on at_risk (AUC drop when shuffled; within-sample, interpretation only):")
print(fi.round(4).to_string(index=False))

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Permutation importance for Random Forest Classifier
permutation importance on at_risk (AUC drop when shuffled; within-sample, interpretation only):
                     feature  auc_drop    std
         impressions_prev30d    0.3900 0.0032
               search_volume    0.3652 0.0044
   main_intent_transactional    0.0810 0.0013
   main_intent_informational    0.0535 0.0018
           has_search_volume    0.0195 0.0006
      main_intent_commercial    0.0190 0.0004
 content_type_feedly article    0.0006 0.0001
content_type_keyword article    0.0006 0.0000
             has_main_intent    0.0001 0.0000
         main_intent_unknown    0.0001 0.0000
    main_intent_navigational    0.0001 0.0000


In [14]:
# wrong-case read on one held-out fold (fold 1)
tr, te = folds[1]
case = df.iloc[te][["content_hash_id", "client_hash_id", "main_intent", "content_type",
                    "search_volume", "impressions_prev30d", "ctr_label", "at_risk"]].copy()
case["p_at_risk_rf"] = fold_scores["rf"][1]
case["baseline_action_score"] = df["baseline_action_score"].to_numpy()[te]

show_cols = ["content_hash_id", "main_intent", "content_type", "search_volume",
             "impressions_prev30d", "ctr_label", "at_risk", "p_at_risk_rf"]

print("MODEL FALSE REVIEWS (predicted at-risk, actually fine / above-median CTR):")
fp = case.sort_values("p_at_risk_rf", ascending=False)
print(fp[fp["at_risk"] == 0].head(3)[show_cols].to_string(index=False))

print("\nMODEL MISSES (actually at-risk, model ranked them lowest risk):")
missed = case.sort_values("p_at_risk_rf", ascending=True)
print(missed[missed["at_risk"] == 1].head(3)[show_cols].to_string(index=False))

MODEL FALSE REVIEWS (predicted at-risk, actually fine / above-median CTR):
         content_hash_id   main_intent    content_type  search_volume  impressions_prev30d  ctr_label  at_risk  p_at_risk_rf
content_03a79c8782deec41 informational keyword article              0               1881.0   0.637980        0           1.0
content_080ef64257742c64 informational keyword article              0               2003.0   3.155368        0           1.0
content_863cb6ff425a8e6e informational keyword article              0               2007.0   1.247723        0           1.0

MODEL MISSES (actually at-risk, model ranked them lowest risk):
         content_hash_id   main_intent    content_type  search_volume  impressions_prev30d  ctr_label  at_risk  p_at_risk_rf
content_0f3935162546a965 informational keyword article             50               8434.0   0.116278        1           0.0
content_3b8c8fe1926c94d9 informational keyword article             10               5535.0   0.159413        1

In [15]:
# final honesty gate: nothing label-window leaks into the scorer, and seeds/versions are pinned
label_window_cols = {"clicks_30d", "impressions_last30d", "avg_position_30d"}
assert not set(feature_cols) & label_window_cols, "leak: label-window column as a feature"
assert "ctr_label" not in feature_cols and "client_hash_id" not in feature_cols and "content_hash_id" not in feature_cols
assert not df["baseline_action_score"].isna().any(), "NaN in baseline score"
print("leakage gate passed — features exist before the label window; IDs and flags excluded")

import sklearn, duckdb, pandas, numpy
print("seeds fixed:", SEED, "| CV folds:", CV)
print("versions: sklearn", sklearn.__version__, "| duckdb", duckdb.__version__,
      "| pandas", pandas.__version__, "| numpy", numpy.__version__)

leakage gate passed — features exist before the label window; IDs and flags excluded
seeds fixed: 42 | CV folds: 5
versions: sklearn 1.6.1 | duckdb 1.3.2 | pandas 2.2.2 | numpy 2.0.2


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.